# 04b. 천리안2A 위성 일사량 (순응 구간 사고만)

입력: `data/3_결과/train_table.csv` (05 결과, 일사량 버그 고친 버전)
출력: `data/1_원본/gk2a_일사량/` (요청별 원본 응답), `data/2_중간/gk2a_solar.csv` (정리본)

**왜 받나**
- ASOS 일사량 = 평균 16.8 km 떨어진 관측소 값 → 사고 자리 하늘과 다를 수 있음
- 측정이 흔들리면 햇빛 배수가 1 쪽으로 끌려가고 95% 범위가 넓어짐
- 천리안2A = 위성으로 추정한 사고 자리 바로 그 점의 일사량

**무엇만 받나**
- A를 쓰는 순응 구간(진입전·경계부·출구부·진출후) 사고만 (약 270건)
- API는 한 번에 최대 24개(30분 간격이면 12시간)만 줌 → 달 전체를 한 번에 못 받음
- 그래서 묶음의 **각 날(사고 날 + 평소 날 3~4개)마다** 사고 시각 앞뒤 1시간(30분 간격 5개)만 요청
- 요청 수 ≈ 순응 구간 낮 묶음 약 200개 × 4~5일 ≈ 900번

**준비 (형)**
1. 기상청 API허브(apihub.kma.go.kr) → "천리안위성 2A호 인공지능 기반 일사량 지점 조회" 활용신청
2. 마이페이지 인증키를 `.env`에 한 줄 추가: `KMA_APIHUB_KEY=발급받은키`
3. 주소·변수 이름은 형이 확인한 예시(11/3)로 이미 채워 둠: `nph_sun_sat_ana_txt`, tm1·tm2·int·lat·lon

In [1]:
%pip install requests -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, time, json, io
from pathlib import Path
from urllib.parse import urlsplit, parse_qs
import numpy as np
import pandas as pd
import requests

ROOT = Path.cwd()
RAW, INTERIM, RESULT = ROOT / "data" / "1_원본", ROOT / "data" / "2_중간", ROOT / "data" / "3_결과"
CACHE = RAW / "gk2a_일사량"
CACHE.mkdir(parents=True, exist_ok=True)

# 형이 11/3 확인한 API 예시 주소
BASE_URL = "https://apihub.kma.go.kr/api/typ01/cgi-bin/url/nph_sun_sat_ana_txt"
EXTRA = {"int": "30"}   # 30분 간격. 예시에 있던 area(지역 코드)는 위경도와 겹쳐서 뺌

def get_key():
    key = None
    if (ROOT / ".env").exists():
        for line in (ROOT / ".env").read_text(encoding="utf-8-sig").splitlines():
            if line.strip().startswith("KMA_APIHUB_KEY="):
                key = line.split("=", 1)[1].strip().strip('"').strip("'")
    key = key or os.environ.get("KMA_APIHUB_KEY")
    assert key, ".env 에 KMA_APIHUB_KEY=... 를 넣어 주세요"
    return key
AUTH_KEY = get_key()
print("키 확인 완료, 주소:", BASE_URL)

키 확인 완료, 주소: https://apihub.kma.go.kr/api/typ01/cgi-bin/url/nph_sun_sat_ana_txt


## 1. 요청 목록 만들기
- 05 결과에서 순응 구간 **낮** 사고의 묶음(사고 날 + 평소 날)을 모두 뽑음 (분석이 낮 사고만 쓰므로)
- 날마다: 그 시각을 30분 단위로 내림 → 1시간 전 ~ 1시간 후 (5개, 한도 24개 안)
- 시간: API는 UTC (11/3 시험 응답으로 확인: 한국 밤 0시 30분 요청 → 값 0) → 한국시간 − 9시간

In [3]:
ZONES = ["진입전", "경계부", "출구부", "진출후"]
t = pd.read_csv(RESULT / "train_table.csv", low_memory=False, parse_dates=["t"])
assert "icsr_stn_id" in t.columns, "05가 옛 버전이에요. 04 → 05 다시 실행 후 진행"
cases = t[(t.y == 1) & t.tunnel_zone.isin(ZONES) & (t.sun_elev > 0)].case_id   # 분석은 낮 사고만 → 밤 사고는 요청 안 함
rows = t[t.case_id.isin(cases) & t.design.isin(["사고", "같은지점"])].copy()   # 사고 날 + 평소 날
rows["t0"] = rows.t.dt.floor("30min")
rows["lat_r"], rows["lon_r"] = rows.lat.round(2), rows.lon.round(2)
rows["file"] = [f"{la:.2f}_{lo:.2f}_{x:%Y%m%d%H%M}.txt" for la, lo, x in zip(rows.lat_r, rows.lon_r, rows.t0)]
req = rows.drop_duplicates("file")[["file", "lat_r", "lon_r", "t0"]].reset_index(drop=True)

def utc_str(ts_kst):
    return (ts_kst - pd.Timedelta(hours=9)).strftime("%Y%m%d%H%M")
req["tm1"] = [utc_str(x - pd.Timedelta(hours=1)) for x in req.t0]
req["tm2"] = [utc_str(x + pd.Timedelta(hours=1)) for x in req.t0]
rows[["case_id", "y", "t", "file"]].to_csv(INTERIM / "gk2a_요청매칭.csv", index=False, encoding="utf-8-sig")
print(f"묶음 {cases.nunique():,}개, 날 {len(rows):,}개 → 요청 {len(req):,}번")

묶음 196개, 날 863개 → 요청 863번


## 2. 시험 요청 3번 (먼저 여기까지만 실행)
- 응답 앞부분이 출력됨 → **이 출력을 Claude에게 보여 주기** (시각 기준·단위 확인, 정리 코드 4번을 맞춤)
- 3번(전체 받기)은 Claude 확인 후에 실행

In [4]:
PARAM = {"start": "tm1", "end": "tm2", "lat": "lat", "lon": "lon"}

def fetch(r):
    params = {**EXTRA,
              PARAM["start"]: r.tm1, PARAM["end"]: r.tm2,
              PARAM["lat"]: f"{r.lat_r:.4f}", PARAM["lon"]: f"{r.lon_r:.4f}",
              "authKey": AUTH_KEY}
    res = requests.get(BASE_URL, params=params, timeout=60)
    res.raise_for_status()
    return res.text

for r in req.head(3).itertuples():
    txt = fetch(r)
    print(r.file, "글자 수", len(txt))
    print(r.tm1, "~", r.tm2, "(UTC로 보낸 값)")
    print(txt[:1500], "\n" + "-" * 60)

36.67_128.13_202301021530.txt 글자 수 704
202301020530 ~ 202301020730 (UTC로 보낸 값)
|      TMFC      |    VAR   |     LON    |    LAT    |  202301020530  |  202301020600  |  202301020630  |  202301020700  |  202301020730  |  
|  202301020530  |  AI-DSR  |  128.1300  |  36.6700  |           nan  |           1.6  |           nan  |           1.0  |           nan  |  
 
------------------------------------------------------------
35.28_128.62_202301311030.txt 글자 수 704
202301310030 ~ 202301310230 (UTC로 보낸 값)
|      TMFC      |    VAR   |     LON    |    LAT    |  202301310030  |  202301310100  |  202301310130  |  202301310200  |  202301310230  |  
|  202301310030  |  AI-DSR  |  128.6200  |  35.2800  |           nan  |           1.1  |           nan  |           1.7  |           nan  |  
 
------------------------------------------------------------
36.28_127.69_202302131130.txt 글자 수 704
202302130130 ~ 202302130330 (UTC로 보낸 값)
|      TMFC      |    VAR   |     LON    |    LAT    |  202302130130 

## 3. 전체 받기 (시험 응답이 정상일 때만)
- 이미 받은 파일은 건너뜀 → 멈춰도 다시 실행하면 이어서
- 오류 응답(인증·호출 한도)이 연속 5번 나오면 멈춤 → 다음 날 다시 실행

In [5]:
fails, done = 0, 0
for r in req.itertuples():
    f = CACHE / r.file
    if f.exists() and f.stat().st_size > 0:
        continue
    txt = None
    for attempt in range(3):
        try:
            txt = fetch(r)
            break
        except Exception as e:
            time.sleep(2 ** attempt)
    if not txt or "error" in txt[:200].lower() or "인증" in txt[:200]:
        fails += 1
        print("실패:", r.file, (txt or "")[:200])
        if fails >= 5:
            print("연속 실패 5번 → 멈춤. 한도 초과면 내일 다시 실행")
            break
        continue
    fails = 0
    f.write_text(txt, encoding="utf-8")
    done += 1
    if done % 100 == 0:
        print(f"{done}번 받음")
    time.sleep(0.2)
have = sum((CACHE / x).exists() for x in req.file)
print(f"받은 요청 {have:,} / {len(req):,}")

받은 요청 863 / 863


## 4. 정리: 한 표로 합치기
**응답 모양 (11/3 확인)**
- 표 한 줄 = 한 지점, 열 = 시각(UTC), 값 = `AI-DSR` (위성 추정 지표 일사량)
- 30분 간격으로 요청해도 값은 **정시에만** 있고 30분은 nan → 사실상 1시간 자료
- 단위: MJ/m² (1시간 합)로 판단. 근거: 2023-01-02 15:41 사고 자리, 16시 값 1.0 ↔ ASOS 일사량 1.05 MJ/m² (W/m²라면 수백이 나와야 함)
- 아래 5번에서 ASOS와 전체 비교로 다시 확인

**붙이는 규칙**: ASOS와 똑같이 "사고 다음 정시" 값 (05의 `tm` 열 시각) → 그 시간(예: 15~16시)을 덮는 값

In [6]:
def parse(txt):
    lines = [l for l in txt.splitlines() if l.strip().startswith("|")]
    head = [c.strip() for c in lines[0].strip("|").split("|")]
    out = []
    for l in lines[1:]:
        v = [c.strip() for c in l.strip("|").split("|")]
        d = dict(zip(head, v))
        for k in head[4:]:
            out.append({"lon": float(d["LON"]), "lat": float(d["LAT"]),
                        "time_kst": pd.to_datetime(k, format="%Y%m%d%H%M") + pd.Timedelta(hours=9),
                        "gk2a_icsr": pd.to_numeric(d[k], errors="coerce")})
    return out

recs = []
for f in req.file:
    p = CACHE / f
    if p.exists():
        for x in parse(p.read_text(encoding="utf-8")):
            x["file"] = f
            recs.append(x)
g = pd.DataFrame(recs).dropna(subset=["gk2a_icsr"]).drop_duplicates(["file", "time_kst"])

m = pd.read_csv(INTERIM / "gk2a_요청매칭.csv", parse_dates=["t"])
m = m.merge(t[["case_id", "y", "t", "tm", "icsr"]], on=["case_id", "y", "t"], how="left")
m["tm"] = pd.to_datetime(m.tm)
m = m.merge(g[["file", "time_kst", "gk2a_icsr"]], left_on=["file", "tm"], right_on=["file", "time_kst"], how="left").drop(columns="time_kst")
m.to_csv(INTERIM / "gk2a_solar.csv", index=False, encoding="utf-8-sig")
print(f"날 {len(m):,}개 중 천리안 값 있음 {m.gk2a_icsr.notna().sum():,}개")
m.head()

날 863개 중 천리안 값 있음 856개


,case_id,y,t,file,tm,icsr,gk2a_icsr
0,4,1,2023-01-02 15:41:00,36.67_128.13_202301021530.txt,2023-01-02 16:00:00,1.05,1.0
1,114,1,2023-01-31 10:30:00,35.28_128.62_202301311030.txt,2023-01-31 11:00:00,1.69,1.7
2,151,1,2023-02-13 11:50:00,36.28_127.69_202302131130.txt,2023-02-13 12:00:00,0.95,0.8
3,154,1,2023-02-13 09:10:00,35.27_128.45_202302130900.txt,2023-02-13 10:00:00,0.09,0.3
4,224,1,2023-03-10 15:59:00,37.24_127.28_202303101530.txt,2023-03-10 16:00:00,1.62,1.7


## 5. 확인: 천리안 vs ASOS
- 같은 날·같은 시각 두 값이 비슷하게 움직이면(상관 0.7 이상) 단위·시각 맞춤이 맞음
- 차이가 큰 날 = 관측소와 사고 자리 하늘이 달랐던 날 → 천리안을 쓰는 이유

In [7]:
ok = m.dropna(subset=["gk2a_icsr", "icsr"])
print("상관:", round(ok.gk2a_icsr.corr(ok.icsr), 3), "| 비교한 날", len(ok))
print("천리안/ASOS 중앙값 비:", round((ok.gk2a_icsr / ok.icsr.replace(0, np.nan)).median(), 2))
print("차이가 0.5 MJ 넘는 날 비율:", round(((ok.gk2a_icsr - ok.icsr).abs() > 0.5).mean(), 3))
ok[["gk2a_icsr", "icsr"]].describe().round(2)

상관: 0.892 | 비교한 날 856
천리안/ASOS 중앙값 비: 1.0
차이가 0.5 MJ 넘는 날 비율: 0.121


,gk2a_icsr,icsr
count,856.00,856.00
mean,1.41,1.35
std,0.98,1.00
min,0.00,0.00
25%,0.50,0.43
50%,1.40,1.28
75%,2.20,2.13
max,3.70,3.67
